# AI Stock Market Research Assistant Agent

This notebook implements an AI agent with the following capabilities:
* Pull current and historical price data for tickers
* Semantic search over news articles and company profiles
* Compare multiple tickers on fundamentals or recent price action
* Add/remove tickers from user watchlists
* Save research notes and analysis reports
* Flag notable price moves or news

The agent uses:
* **Lakebase Postgres** for OLTP storage (users, watchlists, research notes)
* **Vector Search** for semantic retrieval over unstructured text
* **Yahoo Finance API** (via yfinance) for real-time market data
* **LangGraph** for agent orchestration

## Setup Instructions

### Prerequisites

Before running this notebook, ensure you have:

1. **Lakebase Postgres Project**
   - Create a Lakebase project using the Databricks SDK or UI
   - Note the project name, branch name, and database name

2. **Vector Search Endpoint**
   - Create a Vector Search endpoint (Standard or Storage-Optimized)
   - Your index should be created from notebook `03_embed_and_index`

3. **Unity Catalog Resources**
   - Catalog and schema for storing embeddings
   - Proper permissions on the catalog/schema

### Configuration Steps

1. **Update Cell 2 (Configuration)**:
   ```python
   LAKEBASE_PROJECT_NAME = "your-project-name"
   CATALOG = "your_catalog"
   SCHEMA = "your_schema"
   VECTOR_SEARCH_ENDPOINT = "vector_search"
   ```

2. **Run Cell 3**: Install dependencies and restart Python

3. **Run Cell 4**: Import libraries

4. **Run Cell 5**: Initialize connections

5. **Run the schema setup function** (Cell 16):
   ```python
   setup_database_schema()
   ```

6. **Test the agent** (Cell 17): Run the test cell to verify all capabilities

### Agent Capabilities

The agent provides these tools:

* **Market Data**:
  - `get_current_price`: Real-time price quotes
  - `get_historical_prices`: Historical performance
  - `get_company_fundamentals`: Financial metrics

* **Semantic Search**:
  - `search_news_and_filings`: Find relevant news/articles
  - `search_company_profiles`: Discover companies by description

* **Watchlist Management**:
  - `add_to_watchlist`: Add tickers to watchlist
  - `remove_from_watchlist`: Remove tickers
  - `get_watchlist`: View watchlist

* **Research & Analysis**:
  - `save_research_note`: Save analysis notes
  - `get_research_notes`: Retrieve saved notes
  - `compare_tickers`: Side-by-side comparison

### Next Steps

After completing this notebook:

1. **Build a Databricks App**: Create a Streamlit or Dash frontend
2. **Add more tools**: Extend agent capabilities (alerts, portfolio tracking)
3. **Integrate LLM**: Replace rule-based routing with an actual LLM (e.g., Llama, GPT)
4. **Deploy**: Set up the app for production use

### Troubleshooting

**Connection Issues**:
- Verify Lakebase project exists: `w.postgres.list_projects()`
- Check endpoint status: `w.postgres.list_endpoints(parent="...")`
- Ensure OAuth tokens are being generated correctly

**Vector Search Issues**:
- Verify index exists and is synced
- Check endpoint permissions
- Ensure embeddings table has data

**Yahoo Finance Issues**:
- Yahoo Finance data is free and requires no API key
- Rate limits: avoid excessive requests (>2000/hour)
- Some tickers may have delayed or missing data

In [0]:
# Lakebase configuration
LAKEBASE_PROJECT_NAME = "new_database"  # Your actual project name
LAKEBASE_BRANCH_NAME = "production"
LAKEBASE_DATABASE_NAME = "databricks_postgres"

# Vector Search configuration
CATALOG = "stock_research_capstone"  # Replace with your catalog
SCHEMA = "main"   # Replace with your schema
VECTOR_SEARCH_ENDPOINT = "vector_search"  # Your vector search endpoint
VECTOR_INDEX_NAME = f"{CATALOG}.{SCHEMA}.text_embeddings_index"
EMBEDDING_MODEL_ENDPOINT = "databricks-bge-large-en"  # 1024-dim BGE model

# Yahoo Finance API (free, no API key required)
# Note: Massive Stocks API is not available, using Yahoo Finance instead

# Default user for testing
DEFAULT_USER_EMAIL = "bchandra.ry@gmail.com"

In [0]:
%pip uninstall -y psycopg2-binary
%pip install --upgrade databricks-sdk>=0.118.0 langgraph langchain-community langchain-core requests yfinance
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
import psycopg2
import requests
import json
import mlflow.deployments
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
import pandas as pd
import yfinance as yf

# LangGraph imports
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
import operator

w = WorkspaceClient()
print("✓ Libraries imported")

In [0]:
# Initialize Vector Search client
vsc = VectorSearchClient()

# Initialize MLflow deployment client for embeddings
mlflow_client = mlflow.deployments.get_deploy_client("databricks")

# Get Lakebase connection details
def get_lakebase_connection():
    """Get psycopg2 connection to Lakebase Postgres."""
    # Get endpoint details
    endpoints = list(w.postgres.list_endpoints(
        parent=f"projects/{LAKEBASE_PROJECT_NAME}/branches/{LAKEBASE_BRANCH_NAME}"
    ))
    
    if not endpoints:
        raise ValueError(f"No endpoints found for branch {LAKEBASE_BRANCH_NAME}")
    
    endpoint = endpoints[0]  # Use primary endpoint
    host = endpoint.status.hosts.host
    
    # Generate OAuth token (valid for 1 hour)
    token_response = w.postgres.generate_database_credential(
        endpoint=endpoint.name
    )
    
    conn = psycopg2.connect(
        host=host,
        port=5432,
        database=LAKEBASE_DATABASE_NAME,
        user="oauth",
        password=token_response.password,
        sslmode="require"
    )
    return conn

print("✓ Connections initialized")

In [0]:
def get_embedding(text: str) -> List[float]:
    """Generate embedding for a text query."""
    response = mlflow_client.predict(
        endpoint=EMBEDDING_MODEL_ENDPOINT,
        inputs={"input": [text[:8000]]}  # Truncate to model max
    )
    return response["data"][0]["embedding"]

def semantic_search(query: str, doc_type: Optional[str] = None, ticker: Optional[str] = None, top_k: int = 5) -> List[Dict]:
    """Search the vector index for relevant documents."""
    try:
        query_embedding = get_embedding(query)
        
        index = vsc.get_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=VECTOR_INDEX_NAME
        )
        
        # Build filter string for storage-optimized endpoint
        filters = []
        if doc_type:
            filters.append(f"doc_type = '{doc_type}'")
        if ticker:
            filters.append(f"ticker = '{ticker}'")
        
        filter_str = " AND ".join(filters) if filters else None
        
        results = index.similarity_search(
            query_vector=query_embedding,
            columns=["doc_id", "doc_type", "ticker", "title", "text", "metadata_date"],
            filters=filter_str,
            num_results=top_k
        )
        
        docs = []
        for row in results.get("result", {}).get("data_array", []):
            docs.append({
                "doc_id": row[0],
                "doc_type": row[1],
                "ticker": row[2],
                "title": row[3],
                "text": row[4][:500],  # Truncate for display
                "date": row[5],
                "score": row[-1]
            })
        return docs
    except Exception as e:
        print(f"Error in semantic search: {e}")
        return []

def get_yahoo_finance_data(ticker: str) -> Dict:
    """Fetch stock data from Yahoo Finance using yfinance."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        return {
            "ticker": ticker,
            "info": info,
            "error": None
        }
    except Exception as e:
        return {"ticker": ticker, "info": {}, "error": str(e)}

def get_yahoo_historical_data(ticker: str, period: str = "1mo") -> pd.DataFrame:
    """Fetch historical price data from Yahoo Finance."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        return hist
    except Exception as e:
        print(f"Error fetching history for {ticker}: {e}")
        return pd.DataFrame()

print("✓ Helper functions defined")

In [0]:
@tool
def get_current_price(ticker: str) -> str:
    """Get the current price and basic info for a stock ticker.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    
    Returns:
        Current price, change, and volume information
    """
    data = get_yahoo_finance_data(ticker)
    
    if data["error"]:
        return f"Error fetching price for {ticker}: {data['error']}"
    
    info = data["info"]
    current_price = info.get('currentPrice', info.get('regularMarketPrice', 'N/A'))
    previous_close = info.get('previousClose', 'N/A')
    
    if current_price != 'N/A' and previous_close != 'N/A':
        change = current_price - previous_close
        change_pct = (change / previous_close) * 100
    else:
        change = 'N/A'
        change_pct = 'N/A'
    
    result = f"""Current data for {ticker}:
    Price: ${current_price if current_price != 'N/A' else 'N/A'}
    Change: ${change if change != 'N/A' else 'N/A'} ({change_pct if change_pct != 'N/A' else 'N/A':.2f}%)
    Volume: {info.get('volume', 'N/A'):,} if info.get('volume') else 'N/A'
    Market Cap: ${info.get('marketCap', 'N/A'):,} if info.get('marketCap') else 'N/A'
    Day Range: ${info.get('dayLow', 'N/A')} - ${info.get('dayHigh', 'N/A')}
    """
    return result

@tool
def get_historical_prices(ticker: str, days: int = 30) -> str:
    """Get historical price data for a stock.
    
    Args:
        ticker: Stock ticker symbol
        days: Number of days of historical data (default 30)
    
    Returns:
        Summary of price performance over the period
    """
    # Map days to yfinance period
    if days <= 7:
        period = "5d"
    elif days <= 30:
        period = "1mo"
    elif days <= 90:
        period = "3mo"
    elif days <= 180:
        period = "6mo"
    else:
        period = "1y"
    
    hist = get_yahoo_historical_data(ticker, period=period)
    
    if hist.empty:
        return f"No historical data available for {ticker}"
    
    first_price = hist['Close'].iloc[0]
    last_price = hist['Close'].iloc[-1]
    change = ((last_price - first_price) / first_price) * 100
    
    max_price = hist['High'].max()
    min_price = hist['Low'].min()
    avg_volume = hist['Volume'].mean()
    
    start_date = hist.index[0].strftime('%Y-%m-%d')
    end_date = hist.index[-1].strftime('%Y-%m-%d')
    
    result = f"""{ticker} - Historical Performance:
    Period: {start_date} to {end_date} ({len(hist)} trading days)
    Starting Price: ${first_price:.2f}
    Ending Price: ${last_price:.2f}
    Change: {change:+.2f}%
    High: ${max_price:.2f}
    Low: ${min_price:.2f}
    Avg Daily Volume: {avg_volume:,.0f}
    """
    return result

@tool
def get_company_fundamentals(ticker: str) -> str:
    """Get fundamental data for a company.
    
    Args:
        ticker: Stock ticker symbol
    
    Returns:
        Key financial metrics and ratios
    """
    data = get_yahoo_finance_data(ticker)
    
    if data["error"]:
        return f"Error fetching fundamentals for {ticker}: {data['error']}"
    
    info = data["info"]
    
    # Format market cap
    market_cap = info.get('marketCap', 'N/A')
    if market_cap != 'N/A':
        market_cap_str = f"${market_cap:,}"
    else:
        market_cap_str = 'N/A'
    
    result = f"""{ticker} Fundamentals:
    Company: {info.get('longName', info.get('shortName', 'N/A'))}
    Sector: {info.get('sector', 'N/A')}
    Industry: {info.get('industry', 'N/A')}
    Market Cap: {market_cap_str}
    P/E Ratio: {info.get('trailingPE', info.get('forwardPE', 'N/A'))}
    EPS: ${info.get('trailingEps', 'N/A')}
    Dividend Yield: {info.get('dividendYield', 'N/A') if info.get('dividendYield') else 'N/A'}
    52-Week Range: ${info.get('fiftyTwoWeekLow', 'N/A')} - ${info.get('fiftyTwoWeekHigh', 'N/A')}
    """
    return result

print("✓ Market data tools defined")

In [0]:
@tool
def search_news_and_filings(query: str, ticker: Optional[str] = None) -> str:
    """Search for relevant news articles and company filings using semantic search.
    
    Args:
        query: Natural language query (e.g., "interest rate exposure", "earnings reports")
        ticker: Optional ticker to filter results (e.g., 'AAPL')
    
    Returns:
        Relevant news articles and company information
    """
    results = semantic_search(query, ticker=ticker, top_k=5)
    
    if not results:
        return f"No relevant documents found for query: {query}"
    
    output = f"Found {len(results)} relevant documents for: '{query}'\n\n"
    
    for i, doc in enumerate(results, 1):
        output += f"{i}. [{doc['ticker']}] {doc['title']}\n"
        output += f"   Type: {doc['doc_type']} | Date: {doc['date']}\n"
        output += f"   Excerpt: {doc['text'][:200]}...\n"
        output += f"   Relevance: {doc['score']:.4f}\n\n"
    
    return output

@tool
def search_company_profiles(query: str) -> str:
    """Search for companies matching a description or criteria.
    
    Args:
        query: Natural language query (e.g., "regional banks", "EV manufacturers")
    
    Returns:
        Matching company profiles
    """
    results = semantic_search(query, doc_type="company_profile", top_k=5)
    
    if not results:
        return f"No companies found matching: {query}"
    
    output = f"Found {len(results)} companies matching: '{query}'\n\n"
    
    for i, doc in enumerate(results, 1):
        output += f"{i}. {doc['title']} ({doc['ticker']})\n"
        output += f"   {doc['text'][:300]}...\n"
        output += f"   Relevance: {doc['score']:.4f}\n\n"
    
    return output

print("✓ Semantic search tools defined")

In [0]:
@tool
def add_to_watchlist(ticker: str, user_email: str = DEFAULT_USER_EMAIL, watchlist_name: str = "default") -> str:
    """Add a ticker to a user's watchlist.
    
    Args:
        ticker: Stock ticker symbol
        user_email: User's email (default: test user)
        watchlist_name: Name of the watchlist (default: 'default')
    
    Returns:
        Confirmation message
    """
    try:
        conn = get_lakebase_connection()
        cursor = conn.cursor()
        
        # Ensure user exists
        cursor.execute(
            "INSERT INTO users (email, created_at) VALUES (%s, NOW()) ON CONFLICT (email) DO NOTHING",
            (user_email,)
        )
        
        # Get or create watchlist
        cursor.execute(
            "SELECT id FROM watchlists WHERE user_email = %s AND name = %s",
            (user_email, watchlist_name)
        )
        result = cursor.fetchone()
        
        if result:
            watchlist_id = result[0]
        else:
            cursor.execute(
                "INSERT INTO watchlists (user_email, name, created_at) VALUES (%s, %s, NOW()) RETURNING id",
                (user_email, watchlist_name)
            )
            watchlist_id = cursor.fetchone()[0]
        
        # Add ticker to watchlist
        cursor.execute(
            """INSERT INTO watchlist_tickers (watchlist_id, ticker, added_at) 
               VALUES (%s, %s, NOW()) 
               ON CONFLICT (watchlist_id, ticker) DO NOTHING""",
            (watchlist_id, ticker.upper())
        )
        
        conn.commit()
        cursor.close()
        conn.close()
        
        return f"✓ Added {ticker.upper()} to {watchlist_name} watchlist for {user_email}"
    
    except Exception as e:
        return f"Error adding to watchlist: {str(e)}"

@tool
def remove_from_watchlist(ticker: str, user_email: str = DEFAULT_USER_EMAIL, watchlist_name: str = "default") -> str:
    """Remove a ticker from a user's watchlist.
    
    Args:
        ticker: Stock ticker symbol
        user_email: User's email (default: test user)
        watchlist_name: Name of the watchlist (default: 'default')
    
    Returns:
        Confirmation message
    """
    try:
        conn = get_lakebase_connection()
        cursor = conn.cursor()
        
        cursor.execute(
            """DELETE FROM watchlist_tickers 
               WHERE watchlist_id IN (
                   SELECT id FROM watchlists 
                   WHERE user_email = %s AND name = %s
               ) AND ticker = %s""",
            (user_email, watchlist_name, ticker.upper())
        )
        
        deleted = cursor.rowcount
        conn.commit()
        cursor.close()
        conn.close()
        
        if deleted > 0:
            return f"✓ Removed {ticker.upper()} from {watchlist_name} watchlist"
        else:
            return f"{ticker.upper()} was not in {watchlist_name} watchlist"
    
    except Exception as e:
        return f"Error removing from watchlist: {str(e)}"

@tool
def get_watchlist(user_email: str = DEFAULT_USER_EMAIL, watchlist_name: str = "default") -> str:
    """Get all tickers in a user's watchlist.
    
    Args:
        user_email: User's email (default: test user)
        watchlist_name: Name of the watchlist (default: 'default')
    
    Returns:
        List of tickers in the watchlist
    """
    try:
        conn = get_lakebase_connection()
        cursor = conn.cursor()
        
        cursor.execute(
            """SELECT wt.ticker, wt.added_at 
               FROM watchlist_tickers wt
               JOIN watchlists w ON wt.watchlist_id = w.id
               WHERE w.user_email = %s AND w.name = %s
               ORDER BY wt.added_at DESC""",
            (user_email, watchlist_name)
        )
        
        tickers = cursor.fetchall()
        cursor.close()
        conn.close()
        
        if not tickers:
            return f"Watchlist '{watchlist_name}' is empty"
        
        output = f"Watchlist '{watchlist_name}' for {user_email}:\n\n"
        for ticker, added_at in tickers:
            output += f"  • {ticker} (added: {added_at})\n"
        
        return output
    
    except Exception as e:
        return f"Error fetching watchlist: {str(e)}"

print("✓ Watchlist tools defined")

In [0]:
@tool
def save_research_note(ticker: str, note: str, user_email: str = DEFAULT_USER_EMAIL, note_type: str = "analysis") -> str:
    """Save a research note or analysis report for a ticker.
    
    Args:
        ticker: Stock ticker symbol
        note: The research note or analysis content
        user_email: User's email (default: test user)
        note_type: Type of note (e.g., 'analysis', 'thesis', 'alert')
    
    Returns:
        Confirmation with note ID
    """
    try:
        conn = get_lakebase_connection()
        cursor = conn.cursor()
        
        # Ensure user exists
        cursor.execute(
            "INSERT INTO users (email, created_at) VALUES (%s, NOW()) ON CONFLICT (email) DO NOTHING",
            (user_email,)
        )
        
        # Save research note
        cursor.execute(
            """INSERT INTO research_notes (user_email, ticker, note_type, content, created_at) 
               VALUES (%s, %s, %s, %s, NOW()) 
               RETURNING id""",
            (user_email, ticker.upper(), note_type, note)
        )
        
        note_id = cursor.fetchone()[0]
        conn.commit()
        cursor.close()
        conn.close()
        
        return f"✓ Research note saved with ID: {note_id}\nTicker: {ticker.upper()}\nType: {note_type}"
    
    except Exception as e:
        return f"Error saving research note: {str(e)}"

@tool
def get_research_notes(ticker: Optional[str] = None, user_email: str = DEFAULT_USER_EMAIL, limit: int = 10) -> str:
    """Get research notes for a ticker or all notes for a user.
    
    Args:
        ticker: Optional ticker to filter notes
        user_email: User's email (default: test user)
        limit: Maximum number of notes to return (default: 10)
    
    Returns:
        List of research notes
    """
    try:
        conn = get_lakebase_connection()
        cursor = conn.cursor()
        
        if ticker:
            cursor.execute(
                """SELECT id, ticker, note_type, content, created_at 
                   FROM research_notes 
                   WHERE user_email = %s AND ticker = %s
                   ORDER BY created_at DESC 
                   LIMIT %s""",
                (user_email, ticker.upper(), limit)
            )
        else:
            cursor.execute(
                """SELECT id, ticker, note_type, content, created_at 
                   FROM research_notes 
                   WHERE user_email = %s
                   ORDER BY created_at DESC 
                   LIMIT %s""",
                (user_email, limit)
            )
        
        notes = cursor.fetchall()
        cursor.close()
        conn.close()
        
        if not notes:
            filter_msg = f" for {ticker}" if ticker else ""
            return f"No research notes found{filter_msg}"
        
        output = f"Research Notes{' for ' + ticker if ticker else ''}:\n\n"
        for note_id, tkr, note_type, content, created_at in notes:
            output += f"[{note_id}] {tkr} - {note_type}\n"
            output += f"Date: {created_at}\n"
            output += f"{content[:300]}...\n\n"
        
        return output
    
    except Exception as e:
        return f"Error fetching research notes: {str(e)}"

print("✓ Research note tools defined")

In [0]:
@tool
def compare_tickers(tickers: List[str], days: int = 30) -> str:
    """Compare multiple tickers on price performance and fundamentals.
    
    Args:
        tickers: List of ticker symbols to compare
        days: Number of days for price comparison (default: 30)
    
    Returns:
        Comparative analysis of the tickers
    """
    if len(tickers) < 2:
        return "Please provide at least 2 tickers to compare"
    
    comparisons = []
    
    # Map days to yfinance period
    if days <= 7:
        period = "5d"
    elif days <= 30:
        period = "1mo"
    elif days <= 90:
        period = "3mo"
    else:
        period = "6mo"
    
    for ticker in tickers:
        try:
            # Get current info
            data = get_yahoo_finance_data(ticker)
            if data["error"]:
                continue
            
            info = data["info"]
            current_price = info.get('currentPrice', info.get('regularMarketPrice', 0))
            
            # Get historical performance
            hist = get_yahoo_historical_data(ticker, period=period)
            
            if not hist.empty:
                first_price = hist['Close'].iloc[0]
                last_price = hist['Close'].iloc[-1]
                change = ((last_price - first_price) / first_price) * 100
                
                comparisons.append({
                    'ticker': ticker,
                    'current_price': current_price if current_price else last_price,
                    'change_pct': change,
                    'volume': info.get('volume', 0),
                    'market_cap': info.get('marketCap', 0)
                })
        except Exception as e:
            print(f"Error comparing {ticker}: {e}")
            continue
    
    if not comparisons:
        return "Unable to fetch data for comparison"
    
    # Sort by performance
    comparisons.sort(key=lambda x: x['change_pct'], reverse=True)
    
    output = f"Comparison of {len(comparisons)} tickers ({days}-day performance):\n\n"
    
    for i, comp in enumerate(comparisons, 1):
        output += f"{i}. {comp['ticker']}\n"
        output += f"   Current Price: ${comp['current_price']:.2f}\n"
        output += f"   {days}-Day Change: {comp['change_pct']:+.2f}%\n"
        output += f"   Volume: {comp['volume']:,}\n"
        output += f"   Market Cap: ${comp['market_cap']:,}\n\n"
    
    # Add insights
    best = comparisons[0]
    worst = comparisons[-1]
    output += f"Best Performer: {best['ticker']} ({best['change_pct']:+.2f}%)\n"
    output += f"Worst Performer: {worst['ticker']} ({worst['change_pct']:+.2f}%)\n"
    
    return output

print("✓ Comparison tools defined")

In [0]:
# Define agent state
class AgentState(TypedDict):
    messages: Annotated[List, operator.add]
    
# Collect all tools
tools = [
    get_current_price,
    get_historical_prices,
    get_company_fundamentals,
    search_news_and_filings,
    search_company_profiles,
    add_to_watchlist,
    remove_from_watchlist,
    get_watchlist,
    save_research_note,
    get_research_notes,
    compare_tickers
]

print(f"✓ Agent configured with {len(tools)} tools")

In [0]:
def should_continue(state: AgentState):
    """Decide whether to continue or end."""
    last_message = state["messages"][-1]
    
    # If the last message has tool calls, continue to tools
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    
    # Otherwise, end
    return "end"

def call_model(state: AgentState):
    """Call the LLM to decide next action."""
    messages = state["messages"]
    
    # Build system prompt
    system_prompt = """You are an AI Stock Market Research Assistant.

You help users:
- Analyze stocks and market data
- Find relevant news and company information through semantic search
- Manage their stock watchlists
- Compare multiple tickers
- Save research notes and analysis

When a user asks about a company or stock:
1. First, use semantic search to find relevant context (news, filings, company profiles)
2. Then, fetch current market data if needed
3. Provide a comprehensive analysis combining both sources

Always be specific with ticker symbols and provide actionable insights.
"""
    
    # For this demo, we'll simulate the LLM response
    # In production, you would call an actual LLM here (e.g., via Databricks Foundation Models)
    
    last_message = messages[-1]
    
    # Simple rule-based routing for demo
    response_message = AIMessage(
        content="I've processed your request using the available tools.",
        tool_calls=[]
    )
    
    return {"messages": [response_message]}

print("✓ Agent logic defined")

In [0]:
# Create the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

# Set entry point
workflow.set_entry_point("agent")

# Add conditional edges
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)

# After tools, go back to agent
workflow.add_edge("tools", "agent")

# Compile the graph
agent_executor = workflow.compile()

print("✓ Agent graph compiled and ready")

In [0]:
def chat_with_agent(user_message: str):
    """Simple interface to chat with the agent."""
    print(f"\n{'='*80}")
    print(f"USER: {user_message}")
    print(f"{'='*80}\n")
    
    # For demo purposes, directly call tools based on keywords
    # In production, an LLM would decide which tools to use
    
    response = ""
    
    if "price" in user_message.lower() and "current" in user_message.lower():
        # Extract ticker (simple approach)
        words = user_message.split()
        for word in words:
            if word.isupper() and len(word) <= 5:
                response = get_current_price.invoke({"ticker": word})
                break
    
    elif "search" in user_message.lower() or "find" in user_message.lower():
        # Semantic search
        query = user_message.replace("search for", "").replace("find", "").strip()
        response = search_news_and_filings.invoke({"query": query})
    
    elif "add to watchlist" in user_message.lower() or "add" in user_message.lower() and "watchlist" in user_message.lower():
        words = user_message.split()
        for word in words:
            if word.isupper() and len(word) <= 5:
                response = add_to_watchlist.invoke({"ticker": word})
                break
    
    elif "show watchlist" in user_message.lower() or "my watchlist" in user_message.lower():
        response = get_watchlist.invoke({})
    
    elif "compare" in user_message.lower():
        # Extract tickers
        words = user_message.split()
        tickers = [w for w in words if w.isupper() and len(w) <= 5]
        if len(tickers) >= 2:
            response = compare_tickers.invoke({"tickers": tickers})
        else:
            response = "Please provide at least 2 tickers to compare (e.g., 'compare AAPL MSFT GOOGL')"
    
    elif "save note" in user_message.lower() or "research note" in user_message.lower():
        response = "To save a research note, use: save_research_note.invoke({'ticker': 'AAPL', 'note': 'Your analysis here'})"
    
    else:
        response = """I can help you with:
        
• Get current price: "What's the current price of AAPL?"
• Search news: "Search for companies exposed to interest rate risk"
• Manage watchlist: "Add TSLA to my watchlist" or "Show my watchlist"
• Compare stocks: "Compare AAPL MSFT GOOGL"
• Save notes: Use save_research_note tool
• Get fundamentals: Use get_company_fundamentals tool

What would you like to do?"""
    
    print(f"AGENT:\n{response}\n")
    return response

print("\n" + "="*80)
print("AI STOCK MARKET RESEARCH ASSISTANT - READY")
print("="*80)
print("\nExample commands:")
print("  • chat_with_agent('What is the current price of AAPL?')")
print("  • chat_with_agent('Search for companies in the EV sector')")
print("  • chat_with_agent('Add TSLA to my watchlist')")
print("  • chat_with_agent('Compare AAPL MSFT GOOGL')")
print("  • chat_with_agent('Show my watchlist')")

In [0]:
def setup_database_schema():
    """Create the required tables in Lakebase Postgres.
    
    Run this once to set up your database schema.
    """
    try:
        conn = get_lakebase_connection()
        cursor = conn.cursor()
        
        # Create users table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS users (
                email VARCHAR(255) PRIMARY KEY,
                created_at TIMESTAMP DEFAULT NOW(),
                last_login TIMESTAMP
            )
        """)
        
        # Create watchlists table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS watchlists (
                id SERIAL PRIMARY KEY,
                user_email VARCHAR(255) REFERENCES users(email),
                name VARCHAR(100) NOT NULL,
                created_at TIMESTAMP DEFAULT NOW(),
                UNIQUE(user_email, name)
            )
        """)
        
        # Create watchlist_tickers table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS watchlist_tickers (
                id SERIAL PRIMARY KEY,
                watchlist_id INTEGER REFERENCES watchlists(id) ON DELETE CASCADE,
                ticker VARCHAR(10) NOT NULL,
                added_at TIMESTAMP DEFAULT NOW(),
                UNIQUE(watchlist_id, ticker)
            )
        """)
        
        # Create companies table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS companies (
                ticker VARCHAR(10) PRIMARY KEY,
                company_name VARCHAR(255),
                sector VARCHAR(100),
                industry VARCHAR(100),
                company_description TEXT,
                market_cap BIGINT,
                updated_at TIMESTAMP DEFAULT NOW()
            )
        """)
        
        # Create price_snapshots table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS price_snapshots (
                id SERIAL PRIMARY KEY,
                ticker VARCHAR(10) NOT NULL,
                price DECIMAL(10, 2),
                change_pct DECIMAL(5, 2),
                volume BIGINT,
                high DECIMAL(10, 2),
                low DECIMAL(10, 2),
                market_cap BIGINT,
                snapshot_date DATE NOT NULL,
                created_at TIMESTAMP DEFAULT NOW(),
                UNIQUE(ticker, snapshot_date)
            )
        """)
        
        # Create news_articles table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS news_articles (
                id SERIAL PRIMARY KEY,
                ticker VARCHAR(10) NOT NULL,
                title TEXT NOT NULL,
                summary TEXT,
                full_text TEXT,
                source VARCHAR(100),
                url TEXT,
                published_at TIMESTAMP,
                created_at TIMESTAMP DEFAULT NOW()
            )
        """)
        
        # Create research_notes table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS research_notes (
                id SERIAL PRIMARY KEY,
                user_email VARCHAR(255) REFERENCES users(email),
                ticker VARCHAR(10) NOT NULL,
                note_type VARCHAR(50) DEFAULT 'analysis',
                content TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT NOW()
            )
        """)
        
        # Create analysis_reports table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS analysis_reports (
                id SERIAL PRIMARY KEY,
                user_email VARCHAR(255) REFERENCES users(email),
                ticker VARCHAR(10) NOT NULL,
                report_type VARCHAR(50),
                title VARCHAR(255),
                content TEXT,
                generated_by VARCHAR(50) DEFAULT 'agent',
                created_at TIMESTAMP DEFAULT NOW()
            )
        """)
        
        # Create indexes for better performance
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_watchlist_tickers_watchlist ON watchlist_tickers(watchlist_id)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_price_snapshots_ticker_date ON price_snapshots(ticker, snapshot_date DESC)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_news_articles_ticker ON news_articles(ticker, published_at DESC)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_research_notes_user_ticker ON research_notes(user_email, ticker, created_at DESC)")
        
        conn.commit()
        cursor.close()
        conn.close()
        
        print("✓ Database schema created successfully!")
        print("\nTables created:")
        print("  • users")
        print("  • watchlists")
        print("  • watchlist_tickers")
        print("  • companies")
        print("  • price_snapshots")
        print("  • news_articles")
        print("  • research_notes")
        print("  • analysis_reports")
        print("\nYou're ready to use the agent!")
        
    except Exception as e:
        print(f"❌ Error setting up database: {str(e)}")
        print("\nMake sure:")
        print("  1. Your Lakebase project exists")
        print("  2. The project name and branch name are correct in the configuration")
        print("  3. You have the necessary permissions")

# Uncomment the line below to create the schema
# setup_database_schema()

In [0]:
# Example: Test each capability

print("\n" + "="*80)
print("TESTING AGENT CAPABILITIES")
print("="*80)

# 1. Test current price lookup
print("\n1. Testing price lookup...")
try:
    result = get_current_price.invoke({"ticker": "AAPL"})
    print(result)
except Exception as e:
    print(f"Error: {e}")

# 2. Test semantic search for news
print("\n2. Testing semantic search...")
try:
    result = search_news_and_filings.invoke({
        "query": "companies with strong AI capabilities",
        "ticker": None
    })
    print(result)
except Exception as e:
    print(f"Error: {e}")

# 3. Test company profile search
print("\n3. Testing company profile search...")
try:
    result = search_company_profiles.invoke({
        "query": "electric vehicle manufacturers"
    })
    print(result)
except Exception as e:
    print(f"Error: {e}")

# 4. Test watchlist management
print("\n4. Testing watchlist management...")
try:
    # Add a ticker
    result = add_to_watchlist.invoke({"ticker": "AAPL"})
    print(result)
    
    # Show watchlist
    result = get_watchlist.invoke({})
    print(result)
except Exception as e:
    print(f"Error: {e}")

# 5. Test ticker comparison
print("\n5. Testing ticker comparison...")
try:
    result = compare_tickers.invoke({
        "tickers": ["AAPL", "MSFT", "GOOGL"],
        "days": 30
    })
    print(result)
except Exception as e:
    print(f"Error: {e}")

# 6. Test research note saving
print("\n6. Testing research note...")
try:
    result = save_research_note.invoke({
        "ticker": "AAPL",
        "note": "Strong earnings beat expectations. iPhone sales up 15% YoY. Recommend BUY.",
        "note_type": "analysis"
    })
    print(result)
    
    # Retrieve notes
    result = get_research_notes.invoke({"ticker": "AAPL"})
    print(result)
except Exception as e:
    print(f"Error: {e}")

print("\n" + "="*80)
print("TESTING COMPLETE")
print("="*80)

# 🎉 Capstone Project Complete!

## Files Created

✅ **app.py** - Streamlit frontend application  
✅ **app.yaml** - Databricks App configuration  
✅ **README.md** - Complete deployment documentation

## Next Steps to Deploy

### 1. Update Configuration

Edit `app.yaml` with your actual values:
```yaml
LAKEBASE_PROJECT_NAME: "your-project-name"
VECTOR_SEARCH_ENDPOINT: "your-endpoint-name"
VECTOR_INDEX_NAME: "catalog.schema.table"
# Yahoo Finance is used (no API key needed)
```

### 2. Run Database Schema Setup

Scroll up to **Cell 17: Create Database Schema** and uncomment:
```python
setup_database_schema()
```
Then run that cell to create all tables in Lakebase.

### 3. Deploy the App

From a terminal with Databricks CLI:
```bash
cd /Workspace/Users/bchandra.ry@gmail.com/capstone-ai-stock-market

# Create the app
databricks apps create stock-research-assistant --source-code-path .

# Deploy
databricks apps deploy stock-research-assistant
```

### 4. Access Your App

Get the URL:
```bash
databricks apps get stock-research-assistant
```

## App Features

* 💬 **Chat Interface**: Ask questions about stocks and markets
* 📊 **Watchlist Management**: Track your favorite stocks
* 📈 **Live Price Charts**: 90-day candlestick charts
* 🔍 **Semantic Search**: Find relevant news and analysis
* 📝 **Research Notes**: Save your investment thesis
* 🤖 **11 AI Agent Tools**: Market data, search, CRUD operations

## Architecture

```
Streamlit App → AI Agent Tools → Yahoo Finance API
       ↓               ↓
  Lakebase         Vector Search
  (OLTP)           (Semantic RAG)
```

## Requirements Met ✅

1. **Spark Data Pipeline**: Notebook 02 ingests and processes market data
2. **Third-party API**: Yahoo Finance API integration (via yfinance)
3. **Unstructured Data**: Embeddings over news articles and company profiles
4. **Databricks App**: Streamlit frontend with full UX
5. **AI Agent**: 11 tools for read/write operations across data sources

## Troubleshooting

If you encounter issues:
1. Check all configuration values in `app.yaml`
2. Ensure Lakebase project exists and is running
3. Verify Vector Search index is created and synced
4. Test agent tools individually (Cell 18)
5. Check logs: `databricks apps logs stock-research-assistant`

See **README.md** for complete documentation!